In [ ]:
%pip install pandas
%pip install seaborn
%pip install scikit-learn
%pip install matplotlib

In [ ]:
#Importing Required Packages
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn import svm
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
%matplotlib inline


wine = pd.read_csv('./winequality-red.csv', sep = ';') #Update it with a valid path! 


bins = (2, 6.5, 8)  #Examine the impact of number of bins on the model accuracy 
group_names = ['bad', 'good']
wine['quality'] = pd.cut(wine['quality'], bins = bins, labels = group_names)


label_quality = LabelEncoder()


wine['quality']=label_quality.fit_transform(wine['quality'])
wine.head()
# Separate the dataset as response and farure
X = wine.drop('quality', axis = 1)
y = wine['quality']


#Train and test splitting of data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 20, random_state = 42)


sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


#Random Forest Classifier
rfc = RandomForestClassifier(max_depth = 4, n_estimators = 1)
rfc.fit(X_train, y_train)
pred_rfc = rfc.predict(X_test)


print (classification_report(y_test, pred_rfc))
#print (confusion_matrix(y_test, pred_rfc))

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

tree_to_plot = rfc.estimators_[0]

# Plot the decision tree
plt.figure(figsize=(25, 10))
plot_tree(tree_to_plot, feature_names=wine.columns.tolist(), filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree from Random Forest")
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

max_depths = [2, 3, 4, 5, 6, 7, 8, 9, 10]
n_estimators_list = [1, 3, 5, 10, 15, 20, 30, 50]

results = []

# Test all combinations
for max_depth in max_depths:
    for n_est in n_estimators_list:
        rfc_test = RandomForestClassifier(max_depth=max_depth, n_estimators=n_est, random_state=42)
        rfc_test.fit(X_train, y_train)
        pred_test = rfc_test.predict(X_test)
        accuracy = accuracy_score(y_test, pred_test)
        results.append({
            'max_depth': max_depth,
            'n_estimators': n_est,
            'accuracy': accuracy
        })

# Create DataFrame for results
results_df = pd.DataFrame(results)

# Pivot table for visualization
pivot_table = results_df.pivot(index='max_depth', columns='n_estimators', values='accuracy')

print("Accuracy Results for Different Parameter Combinations:\n")
print(pivot_table.to_string())
print(f"\nBest accuracy: {results_df['accuracy'].max():.4f}")
best_params = results_df.loc[results_df['accuracy'].idxmax()]
print(f"Best parameters: max_depth={int(best_params['max_depth'])}, n_estimators={int(best_params['n_estimators'])}")


In [ ]:
# Visualize the results as a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_table, annot=True, fmt='.2f', cmap='YlGnBu', cbar_kws={'label': 'Accuracy'})
plt.title('Model Accuracy vs max_depth and n_estimators')
plt.xlabel('n_estimators (Number of Trees)')
plt.ylabel('max_depth (Tree Depth)')
plt.tight_layout()
plt.show()


### Answer to Question 1:

**Tree Depth:**
- 2-3: Comparatively low accuracy, due to underfitting 
- 4-6: Good to optimal accuracy
- 7+: Optimal accuracy only for high number of estimators. Otherwise low due to overfitting

**Estimator Count:**
- 1: Lowest results
- Higher: Better but diminishing results
- Higher number of estimators can also decrease accuracy
- Needs to be chosen together with tree depth

### Answer to Question 2: 

**Max Depth:**
- Each decision node requires hardware elements 
  - LUT(s) for storing the threshold value 
  - Multiplexers / Comparators for branching 
- Binary classification -> Resource usage is $O(2^D)$ where D is the max depth
- Classification is sequential
  - Traverse top to bottom of tree
  - Latency is linear in tree depth $O(D)$

**Number of Estimators:**
- Can be implemented in parallel or reuse hardware sequentially 
- If parallel: Hardware usage in $O(n)$ where n is the number of estimators
- More trees require additional routing resources and connections 
- If more than one: Voting circuit for final classification 

In [ ]:
FIXED_MAX_DEPTH = 8
FIXED_N_ESTIMATORS = 5

wine_original = pd.read_csv('./winequality-red.csv', sep = ';')
bin_strategies = [
    # 2 classes
    {'bins': (2, 4, 8), 'labels': ['bad', 'good'], 'description': '2-class: bad(3-4), good(5-8)'},
    {'bins': (2, 5, 8), 'labels': ['bad', 'good'], 'description': '2-class: bad(3-5), good(6-8)'},
    {'bins': (2, 6, 8), 'labels': ['bad', 'good'], 'description': '2-class: bad(3-6), good(7-8)'},
    {'bins': (2, 7, 8), 'labels': ['bad', 'good'], 'description': '2-class: bad(3-7), good(8)'},
    
    # 3 classes
    {'bins': (2, 5, 6, 8), 'labels': ['bad', 'medium', 'good'], 'description': '3-class: bad(3-5), medium(5-6), good(6-8)'},
    {'bins': (2, 5, 7, 8), 'labels': ['bad', 'medium', 'good'], 'description': '3-class: bad(3-5), medium(5-7), good(7-8)'},
    {'bins': (2, 4, 6, 8), 'labels': ['bad', 'medium', 'good'], 'description': '3-class: bad(3-4), medium(4-6), good(6-8)'},
    
    # 4 classes
    {'bins': (2, 4, 5, 6, 8), 'labels': ['bad', 'below_avg', 'above_avg', 'good'], 
     'description': '4-class: bad(3-4), below_avg(4-5), above_avg(5-6), good(6-8)'},
    
    # 5 classes
    {'bins': (2, 4, 5, 6, 7, 8), 'labels': ['very_bad', 'bad', 'medium', 'good', 'very_good'], 
     'description': '5-class: very_bad(3-4), bad(4-5), medium(5-6), good(6-7), very_good(7-8)'},
]

bin_results = []

for strategy in bin_strategies:
    # Prepare data with current binning strategy
    wine_test = wine_original.copy()
    wine_test['quality'] = pd.cut(wine_test['quality'], bins=strategy['bins'], labels=strategy['labels'])
    
    # Encode labels
    le_test = LabelEncoder()
    wine_test['quality'] = le_test.fit_transform(wine_test['quality'])
    
    # Split features and target
    X_test_bins = wine_test.drop('quality', axis=1)
    y_test_bins = wine_test['quality']
    
    # Train/test split
    X_train_bins, X_test_bins_split, y_train_bins, y_test_bins_split = train_test_split(
        X_test_bins, y_test_bins, test_size=20, random_state=42
    )
    
    # Scale features
    sc_bins = StandardScaler()
    X_train_bins_scaled = sc_bins.fit_transform(X_train_bins)
    X_test_bins_scaled = sc_bins.transform(X_test_bins_split)
    
    # Train model with fixed parameters
    rfc_bins = RandomForestClassifier(max_depth=FIXED_MAX_DEPTH, n_estimators=FIXED_N_ESTIMATORS, random_state=42)
    rfc_bins.fit(X_train_bins_scaled, y_train_bins)
    pred_bins = rfc_bins.predict(X_test_bins_scaled)
    
    # Calculate accuracy
    accuracy_bins = accuracy_score(y_test_bins_split, pred_bins)
    
    # Store results
    bin_results.append({
        'n_classes': len(strategy['labels']),
        'bins': str(strategy['bins']),
        'description': strategy['description'],
        'accuracy': accuracy_bins,
        'class_distribution': dict(pd.Series(y_test_bins).value_counts().sort_index())
    })

# Create results DataFrame
bin_results_df = pd.DataFrame(bin_results)
print("=" * 100)
print(f"Binning Strategy Results (Fixed: max_depth={FIXED_MAX_DEPTH}, n_estimators={FIXED_N_ESTIMATORS})")
print("=" * 100)
print(bin_results_df[['n_classes', 'description', 'accuracy']].to_string(index=False))
print("\n")
print(f"Best accuracy: {bin_results_df['accuracy'].max():.4f}")
best_bin = bin_results_df.loc[bin_results_df['accuracy'].idxmax()]
print(f"Best binning strategy: {best_bin['description']}")


In [ ]:
# Visualize binning strategy results
plt.figure(figsize=(14, 6))

# Plot 1: Accuracy by number of classes
plt.subplot(1, 2, 1)
grouped = bin_results_df.groupby('n_classes')['accuracy'].agg(['mean', 'max', 'min'])
x_pos = grouped.index
plt.bar(x_pos, grouped['mean'], alpha=0.7, label='Mean Accuracy')
plt.scatter(x_pos, grouped['max'], color='green', s=100, marker='^', label='Max Accuracy', zorder=5)
plt.scatter(x_pos, grouped['min'], color='red', s=100, marker='v', label='Min Accuracy', zorder=5)
plt.xlabel('Number of Classes')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Number of Classes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(x_pos)

# Plot 2: All binning strategies
plt.subplot(1, 2, 2)
colors = plt.cm.viridis(np.linspace(0, 1, len(bin_results_df)))
bars = plt.barh(range(len(bin_results_df)), bin_results_df['accuracy'], color=colors)
plt.yticks(range(len(bin_results_df)), [f"{r['n_classes']}-class" for _, r in bin_results_df.iterrows()])
plt.xlabel('Accuracy')
plt.title('Accuracy for Different Binning Strategies')
plt.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


### Answer to Question 3:

- Accuracy seems to decrease when using more than two classes
- For two and three classes: Threashold selection makes significant difference 
- Best accuracy for the binary intervals (2,4,8) and (2,7,8)
- Accuracy alone not sufficient for selection
- Meaning of quality is also important

In [ ]:
# Define hardware cost model
def calculate_hardware_cost(max_depth, n_estimators):
    """
    Estimate relative hardware cost
    - Each tree has approximately 2^max_depth - 1 nodes
    - Cost is proportional to: n_estimators * (2^max_depth - 1)
    """
    nodes_per_tree = (2 ** max_depth) - 1
    total_nodes = n_estimators * nodes_per_tree
    voting_cost = n_estimators * np.log2(max(n_estimators, 2))
    return total_nodes + voting_cost

hw_friendly_configs = []

# Test configurations 
max_depths_hw = [2, 3, 4, 5, 6]
n_estimators_hw = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Acceptable classifications
bin_labels = [['bad', 'good'], ['bad', 'medium', 'good']]
bins_hw = [
    (2, 5, 8), 
    (2, 6, 8),
    (2, 5, 6, 8),
    (2, 5, 7, 8),
    (2, 4, 6, 8)
]  

for bins_config in bins_hw:
    for max_depth in max_depths_hw:
        for n_est in n_estimators_hw:
            # Prepare data
            wine_hw = pd.read_csv('./winequality-red.csv', sep = ';')
            wine_hw['quality'] = pd.cut(wine_hw['quality'], bins=bins_config, labels=bin_labels[len(bins_config)-3])
            
            le_hw = LabelEncoder()
            wine_hw['quality'] = le_hw.fit_transform(wine_hw['quality'])
            
            X_hw = wine_hw.drop('quality', axis=1)
            y_hw = wine_hw['quality']
            
            X_train_hw, X_test_hw, y_train_hw, y_test_hw = train_test_split(
                X_hw, y_hw, test_size=20, random_state=42
            )
            
            sc_hw = StandardScaler()
            X_train_hw_scaled = sc_hw.fit_transform(X_train_hw)
            X_test_hw_scaled = sc_hw.transform(X_test_hw)
            
            # Train model
            rfc_hw = RandomForestClassifier(max_depth=max_depth, n_estimators=n_est, random_state=42)
            rfc_hw.fit(X_train_hw_scaled, y_train_hw)
            pred_hw = rfc_hw.predict(X_test_hw_scaled)
            
            # Calculate metrics
            accuracy = accuracy_score(y_test_hw, pred_hw)
            hw_cost = calculate_hardware_cost(max_depth, n_est)
            
            # Calculate efficiency metric (accuracy per unit hardware cost)
            efficiency = accuracy / hw_cost if hw_cost > 0 else 0
            
            hw_friendly_configs.append({
                'max_depth': max_depth,
                'n_estimators': n_est,
                'bins': str(bins_config),
                'accuracy': accuracy,
                'hw_cost': hw_cost,
                'efficiency': efficiency,
                'nodes_total': n_est * ((2 ** max_depth) - 1)
            })

# Create DataFrame
hw_configs_df = pd.DataFrame(hw_friendly_configs)

# Display results sorted by efficiency
print("=" * 110)
print("Hardware-Friendly Configurations (Binary Classification with bins=(2, 6.5, 8))")
print("=" * 110)
print(hw_configs_df.sort_values('efficiency', ascending=False)[
    ['max_depth', 'n_estimators', 'accuracy', 'nodes_total', 'hw_cost', 'efficiency']
].to_string(index=False))

print("\n" + "=" * 110)
print("Top 5 Most Efficient Configurations:")
print("=" * 110)
top_5 = hw_configs_df.sort_values('efficiency', ascending=False).head(5)
for idx, row in top_5.iterrows():
    print(f"\nRank {list(top_5.index).index(idx) + 1}:")
    print(f"  max_depth={int(row['max_depth'])}, n_estimators={int(row['n_estimators'])}")
    print(f"  Accuracy: {row['accuracy']:.4f}")
    print(f"  Total nodes: {int(row['nodes_total'])}")
    print(f"  HW Cost (relative): {row['hw_cost']:.1f}")
    print(f"  Efficiency: {row['efficiency']:.6f}")
    print(f"  Binning strategy: {row['bins']}")


In [ ]:
# Visualize hardware/accuracy trade-off
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Accuracy vs Hardware Cost
ax1 = axes[0, 0]
scatter = ax1.scatter(hw_configs_df['hw_cost'], hw_configs_df['accuracy'], 
                      c=hw_configs_df['max_depth'], s=hw_configs_df['n_estimators']*20,
                      cmap='viridis', alpha=0.6, edgecolors='black')
ax1.set_xlabel('Hardware Cost (relative)', fontsize=11)
ax1.set_ylabel('Accuracy', fontsize=11)
ax1.set_title('Accuracy vs Hardware Cost\n(color=max_depth, size=n_estimators)', fontsize=12)
ax1.grid(True, alpha=0.3)
cbar1 = plt.colorbar(scatter, ax=ax1)
cbar1.set_label('max_depth', fontsize=10)

# Plot 2: Efficiency by configuration
ax2 = axes[0, 1]
pivot_eff = hw_configs_df.pivot_table(index='max_depth', columns='n_estimators', values='efficiency', aggfunc='max')
sns.heatmap(pivot_eff, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax2, cbar_kws={'label': 'Efficiency'})
ax2.set_title('Efficiency (Accuracy/HW_Cost)', fontsize=12)
ax2.set_xlabel('n_estimators', fontsize=11)
ax2.set_ylabel('max_depth', fontsize=11)

# Plot 3: Pareto front
ax3 = axes[1, 0]
# Sort by hardware cost
sorted_df = hw_configs_df.sort_values('hw_cost')
ax3.plot(sorted_df['hw_cost'], sorted_df['accuracy'], 'o-', alpha=0.5, label='All configs')

# Find Pareto optimal points (maximize accuracy, minimize cost)
pareto_points = []
max_accuracy_so_far = 0
for _, row in sorted_df.iterrows():
    if row['accuracy'] >= max_accuracy_so_far:
        pareto_points.append(row)
        max_accuracy_so_far = row['accuracy']

pareto_df = pd.DataFrame(pareto_points)
ax3.plot(pareto_df['hw_cost'], pareto_df['accuracy'], 'ro-', linewidth=2, markersize=10, label='Pareto optimal')
ax3.set_xlabel('Hardware Cost (relative)', fontsize=11)
ax3.set_ylabel('Accuracy', fontsize=11)
ax3.set_title('Pareto Front: Optimal Trade-offs', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Accuracy by depth and estimators
ax4 = axes[1, 1]
pivot_acc = hw_configs_df.pivot_table(index='max_depth', columns='n_estimators', values='accuracy', aggfunc='max')
sns.heatmap(pivot_acc, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax4, cbar_kws={'label': 'Accuracy'})
ax4.set_title('Accuracy by Configuration', fontsize=12)
ax4.set_xlabel('n_estimators', fontsize=11)
ax4.set_ylabel('max_depth', fontsize=11)

plt.tight_layout()
plt.show()

# Print Pareto optimal configurations
print("\n" + "=" * 110)
print("Pareto Optimal Configurations (Best accuracy for each hardware cost level):")
print("=" * 110)
print(pareto_df[['max_depth', 'n_estimators', 'accuracy', 'nodes_total', 'hw_cost']].to_string(index=False))


In [ ]:
# Choose most efficient confing that meets minimum accuracy requirement
min_accuracy = 0.85

feasible_configs = hw_configs_df[hw_configs_df['accuracy'] >= min_accuracy] 
if not feasible_configs.empty:
    best_feasible = feasible_configs.sort_values('efficiency', ascending=False).iloc[0]
    print("\n" + "=" * 110)
    print(f"Best Hardware-Friendly Configuration with Accuracy >= {min_accuracy:.2f}:")
    print("=" * 110)
    print(f"max_depth={int(best_feasible['max_depth'])}, n_estimators={int(best_feasible['n_estimators'])}")
    print(f"Accuracy: {best_feasible['accuracy']:.4f}")
    print(f"Total nodes: {int(best_feasible['nodes_total'])}")
    print(f"HW Cost (relative): {best_feasible['hw_cost']:.1f}")
    print(f"Efficiency: {best_feasible['efficiency']:.6f}")
    print(f"Binning strategy: {best_feasible['bins']}")
else:
    print(f"\nNo configurations found with accuracy >= {min_accuracy:.2f}. Consider lowering the threshold or reviewing the results.")


### Answer to Question 4:

**Assumption / Model:**
- Only a selection of 2 and 3 class binning strategies are searched
   - Others might be more accurate / efficient, but not meaningful for the dataset
   - If other binning strategies are reasonable, add them to the search config
- Hardware cost estimation
   - Exponential in max depth 
   - linear in estimators
   - Additional logarithmic term in estimator count  
- Minimal required accuracy
   - Size of classifier is stronger than accuracy
   - Smaller is usually way more efficient than more accurate 
   - For reasonable tradeoff a lower limit should be used

**Results:**
- Max Depth: 3
- N Estimators: 4
- Accuracy: 85%
- Binning: Binary bad: [3,5], good: [6,8]
- Relative HW Cost: 36
- Efficiency: 0.0236

In [ ]:
# Train the optimal model for hardware implementation
OPTIMAL_MAX_DEPTH = 3
OPTIMAL_N_ESTIMATORS = 4
OPTIMAL_BINS = (2, 5, 8)

# Prepare data with optimal configuration
wine_final = pd.read_csv('./winequality-red.csv', sep = ';')
wine_final['quality'] = pd.cut(wine_final['quality'], bins=OPTIMAL_BINS, labels=['bad', 'good'])

le_final = LabelEncoder()
wine_final['quality'] = le_final.fit_transform(wine_final['quality'])

X_final = wine_final.drop('quality', axis=1)
y_final = wine_final['quality']

# Get feature names (scaled features will have same indices)
feature_names = list(X_final.columns)
print("Features used in the model:")
for i, name in enumerate(feature_names):
    print(f"  Feature {i}: {name}")

# Train/test split
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_final, y_final, test_size=20, random_state=42
)

# Scale features
sc_final = StandardScaler()
X_train_final_scaled = sc_final.fit_transform(X_train_final)
X_test_final_scaled = sc_final.transform(X_test_final)

# Store scaling parameters for hardware use
print("\nFeature Scaling Parameters (mean and std for each feature):")
for i, name in enumerate(feature_names):
    print(f"  {name}: mean={sc_final.mean_[i]:.4f}, std={sc_final.scale_[i]:.4f}")

# Train the final model
rfc_final = RandomForestClassifier(max_depth=OPTIMAL_MAX_DEPTH, 
                                   n_estimators=OPTIMAL_N_ESTIMATORS, 
                                   random_state=42)
rfc_final.fit(X_train_final_scaled, y_train_final)

# Evaluate
pred_final = rfc_final.predict(X_test_final_scaled)
accuracy_final = accuracy_score(y_test_final, pred_final)

print(f"\nFinal Model Performance:")
print(f"  Configuration: max_depth={OPTIMAL_MAX_DEPTH}, n_estimators={OPTIMAL_N_ESTIMATORS}")
print(f"  Bins: {OPTIMAL_BINS}")
print(f"  Test Accuracy: {accuracy_final:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test_final, pred_final, target_names=['bad', 'good']))


In [ ]:
for i in range(OPTIMAL_N_ESTIMATORS):
    tree_final = rfc_final.estimators_[i]
    plt.figure(figsize=(30, 15))
    plot_tree(
        tree_final, 
        feature_names=feature_names, 
        class_names=['bad', 'good'],
        filled=True, 
        rounded=True, 
        fontsize=12,
        proportion=True
    )
    plt.title(f"Decision Tree for Hardware Implementation\n(max_depth={OPTIMAL_MAX_DEPTH}, n_estimators={OPTIMAL_N_ESTIMATORS})", 
          fontsize=16, pad=20)
    plt.tight_layout()
    plt.show()


In [ ]:
from typing import TypedDict

class Node(TypedDict):
    node_id: int
    threshold: float
    quantized_threshold: str
    is_leaf: bool
    left_idx: int
    right_idx: int
    prediction: int
    feature_idx: int

def quantize_threshold(node_threshold: float):
    fixed_val = 0
    if node_threshold >= 0:
        fixed_val = int(node_threshold * 65536) & 0xFFFFFFFF
    else:
        # Two's complement for negative values
        fixed_val = (int(abs(node_threshold) * 65536) ^ 0xFFFFFFFF) + 1
        fixed_val = fixed_val & 0xFFFFFFFF

    return fixed_val

def build_node(node: Node) -> str:
    if node['is_leaf']:
        return f"""
{node['node_id']}: begin
    prediction <= {node['prediction']};
    done <= 1;
end
"""
    else:
        return f"""
{node['node_id']}: begin
    if (features[{node['feature_idx']}] <= 32'sh{node['quantized_threshold'][2:]}) begin
        current_node <= {node['left_idx']};
    end else begin
        current_node <= {node['right_idx']};
    end
end
"""

def build_verilog(id: int, nodes: list[Node]) -> str:
    lines = [line for line in [build_node(node) for node in nodes] for line in line.splitlines()]
    lines = list(filter(lambda x: x.strip() != "", lines))
    for idx in range(len(lines)):
        lines[idx] = "\n" + " " * 10 + lines[idx]

    return f"""
module tree_{id}(
  input wire clk,
  input wire reset,
  input wire start_traversal,
  input wire signed [31:0] features [0:10],  
  output reg prediction,      
  output reg done             
);
  
  reg [4:0] current_node;     
  always @(posedge clk or negedge reset) begin
    if (!reset) begin
      current_node <= 5'd0;
      done <= 1'b0;
      prediction <= 1'b0;
    end else begin
      if (start_traversal && !done) begin
        case (current_node)
          {"".join(lines)}
        endcase
      end
    end
  end
endmodule
"""

trees = len(rfc_final.estimators_)
for tree_id in range(trees):
    tree = rfc_final.estimators_[tree_id]
    nodes = tree.tree_.node_count
    node_configs: list[Node] = []
   
    for node in range(nodes):
        threshold = float(tree.tree_.threshold[node])
        quantized_threshold = quantize_threshold(threshold)
        is_leaf = bool(tree.tree_.children_left[node] == tree.tree_.children_right[node])
        left_idx = int(tree.tree_.children_left[node])
        right_idx = int(tree.tree_.children_right[node])
        prediction = int(tree.tree_.value[node][0].argmax()) if is_leaf else -1
        feature_idx = int(tree.tree_.feature[node])
        node_configs.append({
            "node_id": node,
            "threshold": threshold,
            "quantized_threshold": f"{quantized_threshold:#010x}",  
            "is_leaf": is_leaf,
            "left_idx": left_idx,
            "right_idx": right_idx,
            "prediction": prediction,
            "feature_idx": feature_idx
        })

    verilog = build_verilog(tree_id, node_configs)
    print(verilog)

In [ ]:
def print_test_case_values(X_test, y_test, num_cases):
    for i in range(min(num_cases, len(X_test))):
        features = X_test[i]
        label = y_test[i]
        print(f"Test Case {i+1}:")
        for idx, feature_value in enumerate(features):
            quant = f"{quantize_threshold(feature_value):#010x}"
            print(f"features[{idx}] <= 32'sh{quant[2:]};")
        print(f"expected_pred <= 1'b{label};\n")


print_test_case_values(X_test_final_scaled, y_test_final.values, 3)

### Verilog

Run the cells above and copy the code for the four sub-trees to RFC_HW.sv.
The values for the test cases can be copied to the testbench. 
Additional test cases can be generated by changing num_cases.

To verify the design run 

```sh
make compile
make simulate
```